In [ ]:
# programas necessarios para o build do apk
%%bash

set -e

echo "🔵 Atualizando sistema..."
apt update -qq

echo "🔵 Instalando Java 17..."
apt install -y openjdk-17-jdk wget unzip > /dev/null

echo "🔵 Instalando Gradle 9.3.1..."
rm -rf /opt/gradle
wget -q https://services.gradle.org/distributions/gradle-9.3.1-bin.zip
unzip -q gradle-9.3.1-bin.zip
mv gradle-9.3.1 /opt/gradle

echo "🔵 Instalando Android SDK..."
rm -rf /content/android-sdk
mkdir -p /content/android-sdk/cmdline-tools

wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
unzip -q commandlinetools-linux-11076708_latest.zip -d /content/android-sdk/cmdline-tools

# Corrigir estrutura obrigatória
mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest

echo "🔵 Configurando variáveis de ambiente..."

export ANDROID_HOME=/content/android-sdk
export ANDROID_SDK_ROOT=/content/android-sdk
export GRADLE_HOME=/opt/gradle
export PATH=$GRADLE_HOME/bin:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools:$ANDROID_HOME/build-tools/36.0.0:$PATH

echo "🔵 Aceitando licenças..."
yes | sdkmanager --licenses > /dev/null

echo "🔵 Instalando Android 36 + Build Tools 36..."
sdkmanager "platform-tools" \
           "platforms;android-36" \
           "build-tools;36.0.0"

echo "🔵 Criando debug.keystore global padrão..."

keytool -genkeypair \
  -v \
  -keystore /root/.android/debug.keystore \
  -storepass android \
  -keypass android \
  -alias androiddebugkey \
  -keyalg RSA \
  -keysize 2048 \
  -validity 10000 \
  -dname "CN=Android Debug,O=Android,C=US"

echo "✅ INSTALAÇÃO COMPLETA!"
echo "--------------------------------------"
echo "Java:"
java -version
echo "--------------------------------------"
echo "Gradle:"
gradle -v
echo "--------------------------------------"
echo "SDK Manager:"
sdkmanager --version
echo "--------------------------------------"

In [ ]:
# upload projeto do google ai studio (.zip do apk do google ai studio)
from google.colab import files
uploaded = files.upload()

In [ ]:
# descompacte o zip do projeto apk do google ai studio
!unzip oneplay.zip

In [ ]:
# gerar debug.keystore na pasta oneplay mude para sua pasta do zip
%%bash

cd oneplay

echo "🔐 Criando debug.keystore dentro do projeto..."

keytool -genkeypair \
  -v \
  -keystore debug.keystore \
  -storepass android \
  -keypass android \
  -alias androiddebugkey \
  -keyalg RSA \
  -keysize 2048 \
  -validity 10000 \
  -dname "CN=Android Debug,O=Android,C=US"

echo "✅ debug.keystore criado em oneplay/"

In [ ]:
# compilar apk
%%bash

export ANDROID_HOME=/content/android-sdk
export ANDROID_SDK_ROOT=/content/android-sdk
export GRADLE_HOME=/opt/gradle
export PATH=$GRADLE_HOME/bin:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools:$PATH

cd oneplay
echo "sdk.dir=/content/android-sdk" > local.properties

stdbuf -oL -eL gradle assembleDebug --no-daemon --console=plain

In [6]:
# baixar apk
from google.colab import files
files.download("oneplay/app/build/outputs/apk/debug/app-debug.apk")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>